# Customer-Base Audit

Derived from *The Customer-Base Audit: An Excel-Based Companion* (Fader, Hardie, Ross, v1.0).

Quarterly aggregated data is provided by **Madrigal** and represents a 1% sample of data from **Q1 2016 - Q4 2019** (16 quarters) of **70,041 customers**.

Product-dimension analyses (TCBA Ch. 8) are out of scope.

## Imports

In [1]:
from functools import reduce
import pandas as pd
from great_tables import GT, loc, style

## Data

### Long Format

`cust_data_long.csv` — one row per customer × quarter *in which the customer was active*. ~130,789 rows.

| Column | Meaning |
|---|---|
| `CustomerID` | customer key |
| `Cohort` | acquisition quarter, e.g. `y2016 q1`; also `pre y2016` for customers acquired before the observation window |
| `YearQuarter` | `y2016 q1` … `y2019 q4` |
| `NumTrans` | transactions in that quarter |
| `Spend` | revenue in that quarter |
| `Profit` | contribution profit in that quarter |

Derive `Year` from the first 5 characters of `YearQuarter` (`y2016` … `y2019`).

In [2]:
cust_data = pd.read_csv("data/madrigal/cust_data_long.csv")
cust_data = (
    cust_data
    .assign(
        **cust_data["YearQuarter"]
        .str.extract(r"y(\d{4})_q(\d)")
        .rename(columns={0: "Year", 1: "Quarter"})
        .astype({"Year": "int32", "Quarter": "int8"})
    )
    .drop(columns="YearQuarter")
)

### Wide to Long format

Three files — `cust_by_qtr_trans.csv`, `cust_by_qtr_spend.csv`, `cust_by_qtr_profit.csv` — one row per customer, one column per quarter, plus a `Cohort` column. Mostly zeros/blanks. If you use these, **do not assume the three files share the same CustomerID ordering — verify.**

In this exercise, we will be using the long format data. However, if you only have wide format data, you can create a long format dataframe with the following steps:

```python
def wide_to_long(wide_df, value):
    long_data = wide_df.melt(
        id_vars=["CustomerID", "Cohort"], 
        value_vars=wide_df.columns[2:], 
        var_name="YearQuarter", 
        value_name=value
    ).sort_values(
        ["CustomerID", "YearQuarter"]
    )
    
    if value == "NumTrans":
        long_data = long_data.query(
            "NumTrans > 0"
        ).astype({"NumTrans": "int32"})
    
    return long_data.reset_index(drop=True)

trans_wide = pd.read_csv("data/madrigal/cust_by_qtr_trans.csv")
spend_wide = pd.read_csv("data/madrigal/cust_by_qtr_spend.csv")
profit_wide = pd.read_csv("data/madrigal/cust_by_qtr_profit.csv")

cust_data_long = reduce(
    lambda left, right: left.merge(
        right,
        on=["CustomerID", "Cohort", "YearQuarter"],
        how="left",
    ),
    (
        wide_to_long(df, value)
        for df, value in [
            (trans_wide, "NumTrans"),
            (spend_wide, "Spend"),
            (profit_wide, "Profit"),
        ]
    ),
)
```

### Data Conventions

- **Cohort** = customers acquired in a given period (quarter or year).
- **Cohort size** = number of customers whose acquisition period is that period (diagonal of a cohort × period active-customer matrix). Size of the `pre y2016` cohort is **unknown** — exclude it from any %-active or per-cohort-size calculation.
- **AOF** (average order frequency) = total transactions ÷ number of active customers.
- **AOV** (average order value) = total spend ÷ total transactions.
- **Average margin** = total profit ÷ total spend.
- Core multiplicative decomposition of profit:

  ```
  Profit = #customers × AOF × AOV × Margin
         = #customers × (trans/cust) × (spend/trans) × (profit/spend)
  ```

- For cohorts in a period, the decomposition extends to:

  ```
  Cohort profit = cohort size × % cohort active × AOF × AOV × Margin
  Cohort revenue = cohort size × % cohort active × ASPAC
  ASPAC (avg spend per active cohort member) = AOF × AOV
  ```

### Histogram Binning Conventions

- Bins are **half-open on the left**: the `$25–50` bin counts spend in `(25, 50]`. The first bin is inclusive of its lower edge, so a customer with $0 spend falls in the first bin.
- Behavioural distributions are heavily right-skewed (max often 10–100× the mean), so every histogram gets a **right-censoring point** — a terminal `> x` bin.
- Choose bin width from the percentile table, not by rule of thumb. Preferred widths: 1, 2, 5, 10, 20, 25, 50, 100, 200, 250, 500. Too narrow → noisy; too wide → hides the skew.
- Choose the censoring point so the final bin isn't overloaded. If ~5% of customers exceed $578, censoring at $600 puts too much mass in the last bin; $1000 is better.
- Plot **relative frequencies**, not counts, whenever you compare two groups of different size.
- In Python, prefer `np.histogram` with explicit `bins=` edges, or `pd.cut(..., right=True)`. (The Excel `FREQUENCY` warning in the book is not relevant to you.)

## Lens 1 — How do customers differ from one another? (single period)

**Objective:** quantify variability in buying behaviour across customers within one calendar year (e.g. 2019). Answer: the "average customer" is a fiction.

### Working dataset
Filter long data to `Year == y2019`, group by `CustomerID`, sum `NumTrans`, `Spend`, `Profit`. Only customers with ≥1 transaction in 2019 appear.

**Validation targets**

| Metric | Value |
|---|---|
| Active customers, 2019 | 31,855 |
| Total transactions | 60,730 |
| Total spend | ≈ $5,836,712 |
| Total profit | ≈ $2,798,904 |
| Avg transactions / active customer | 1.9 |
| Avg spend / active customer | $183 |
| Avg profit / active customer | $88 |

In [7]:
year = 2019

cust_data_latest = (
    cust_data
    .query(f"Year == {year}")
    .groupby("CustomerID", as_index=False)
    .agg(
        NumTrans=("NumTrans", "sum"),
        Spend=("Spend", "sum"),
        Profit=("Profit", "sum"),
    )
)

summary = {
    "Active Customers": len(cust_data_latest),
    "Total Transactions": cust_data_latest["NumTrans"].sum(),
    "Total Spend": cust_data_latest["Spend"].sum(),
    "Total Profit": cust_data_latest["Profit"].sum(),
    "Avg. Transactions / Active Customer": cust_data_latest["NumTrans"].mean(),
    "Avg. Spend / Active Customer": cust_data_latest["Spend"].mean(),
    "Avg. Profit / Active Customer": cust_data_latest["Profit"].mean(),
}

summary = (
    pd.DataFrame(summary.items(), columns=["Metric", "Value"])
)

(
    GT(summary)
    .tab_header(
        title=f"{year} Annual Customer Summary",
        subtitle="Customer activity, revenue, and profitability metrics"
    )
    .fmt_number(
        columns="Value",
        rows=[0, 1, 2, 3],
        decimals=0
    )
    .fmt_currency(
        columns="Value",
        rows=[4, 5, 6],
        decimals=2
    )
    .fmt_number(
        columns="Value",
        rows=[4],
        decimals=2
    )
)

GT(_tbl_data=                                Metric         Value
0                     Active Customers  3.185500e+04
1                   Total Transactions  6.073000e+04
2                          Total Spend  5.836712e+06
3                         Total Profit  2.798904e+06
4  Avg. Transactions / Active Customer  1.906451e+00
5         Avg. Spend / Active Customer  1.832275e+02
6        Avg. Profit / Active Customer  8.786387e+01, _body=<great_tables._gt_data.Body object at 0x71e62e93bcb0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x71e62e93c9d0>, _spanners=Spanners([]), _heading=Heading(title='2019 Annual Customer Summary', subtitle='Customer activity, revenue, and profitability metrics', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x71e62e9a1650>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x71e62e9a1750>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x71e62e93e520>, _formats=[<great_tables._gt_data.FormatInfo object at 0x71e62e93ba10>, <great_tables._gt_data.FormatInfo object at 0x71e62e9b82f0>, <great_tables._gt_data.FormatInfo object at 0x71e62e91f380>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, catego

### Distribution of spend

Descriptives first (min, max, mean, median, % below mean), then percentiles at 5% intervals (5%…95%), then bin.

**Findings:** max $6,695 ≈ 37× the mean. Mean $183 > median $113. **69% of customers spend below average.** 5th pct $22.20; 10th pct $30.22; the top 5% each spent more than $578.52.

**Bins:** width $25, censor at $1000 → 41 bins. (Width $20 → 51 bins, also defensible.)

**Note:** two customers have exactly $0 spend in 2019; they land in the first bin.

In [16]:
cols = ["NumTrans", "Spend", "Profit"]

summary = {
    "Count": [cust_data_latest[col].count() for col in cols],
    "Mean": [cust_data_latest[col].mean() for col in cols],
    "Std. Dev.": [cust_data_latest[col].std() for col in cols],
    "Min": [cust_data_latest[col].min() for col in cols],
    "5%": [cust_data_latest[col].quantile(0.05) for col in cols],
    "10%": [cust_data_latest[col].quantile(0.1) for col in cols],
    "25%": [cust_data_latest[col].quantile(0.25) for col in cols],
    "50%": [cust_data_latest[col].quantile(0.5) for col in cols],
    "75%": [cust_data_latest[col].quantile(0.75) for col in cols],
    "max%": [cust_data_latest[col].max() for col in cols],
}
summary

{'Count': [np.int64(31855), np.int64(31855), np.int64(31855)],
 'Mean': [np.float64(1.9064511065766756),
  np.float64(183.2275109088055),
  np.float64(87.8638731753257)],
 'Std. Dev.': [np.float64(2.0740908763710606),
  np.float64(239.39117494027136),
  np.float64(118.39832637452295)],
 'Min': [np.int64(1), np.float64(0.0), np.float64(-651.62)],
 '5%': [np.float64(1.0), np.float64(22.197), np.float64(7.697000000000001)],
 '10%': [np.float64(1.0), np.float64(30.22), np.float64(12.474000000000002)],
 '25%': [np.float64(1.0), np.float64(57.59), np.float64(24.76)],
 '50%': [np.float64(1.0), np.float64(113.95), np.float64(52.4)],
 '75%': [np.float64(2.0), np.float64(216.37), np.float64(105.96000000000001)],
 'max%': [np.int64(58), np.float64(6695.02), np.float64(3347.09)]}

,CustomerID,NumTrans,Spend,Profit
count,31855.000000,31855.000000,31855.000000,31855.000000
mean,45036.915272,1.906451,183.227511,87.863873
std,21616.009651,2.074091,239.391175,118.398326
min,4.000000,1.000000,0.000000,-651.620000
25%,28598.500000,1.000000,57.590000,24.760000
50%,54114.000000,1.000000,113.950000,52.400000
75%,62077.500000,2.000000,216.370000,105.960000
max,70041.000000,58.000000,6695.020000,3347.090000


### 1.2 Distribution of profit
Same structure, but with an explicit **`< $0` bin** for loss-making customers.

**Findings:** range –$652 to $3,347. Mean $88 > median $54; again **69% below average**. 5th pct $7.70; 10th pct $12.47; top 5% above $282.45.

**Bins:** width $25, censor at $500, plus the `<0` bin. (Profit mean/median/max run ~45–50% of the corresponding spend figures, which is what motivates the lower cut-off.)

### 1.3 Distribution of number of transactions
Simple value-count of `NumTrans`.

**Findings:** right-skewed (reverse-J). Max = 58 transactions. **63% of customers made exactly one purchase** (20,149 / 31,855), so the mean of 1.9 is misleading.

**Bins:** width 1, censor at `10+`. (Censoring at 5 is too low — 6.8% of customers made ≥5 transactions.) Keep all non-terminal bins **equal width**; do not do 1 / 2–4 / 5–9 / 10+. If you want unequal groupings, use a table, not a histogram.

### 1.4 Distribution of average spend per transaction
Per customer: `avg_spend = Spend / NumTrans`.

**Bins:** width $25, censor at $500. ($20 / $300 also reasonable; the $500 cut-off shows the tail better.)

#### Important: two different "average spend per transaction" numbers
These are **not** the same quantity and both appear in practice:

- **Unweighted mean of the per-customer average** = `mean(spend_i / trans_i)` = **$99**
- **AOV** = `sum(spend_i) / sum(trans_i)` = **$96**

The second is the transaction-weighted version of the first:

```
AOV = Σ [ trans_i / Σtrans ] × ( spend_i / trans_i )
```

They coincide only if every customer made the same number of transactions. Note that $183 / 1.9 gives the AOV ($96), not the mean of customer averages ($99). **Always know which one a report is showing.** Use "AOV" strictly for the weighted version.

### 1.5 Average spend per transaction, by transaction level
Group customers by transaction count (1, 2, …, 9, `10+` — same bins as §1.3). For each group report: **mean, std dev, min, max, 5th pct, median, 95th pct** of per-customer average spend. Purpose: is average order value related to purchase frequency?

### 1.6 Distribution of average margin
Per customer: `margin = Profit / Spend`, defined only where `Spend > 0` (two customers have zero spend — exclude them / set NaN, don't zero-fill).

**Caveat:** this is the *overall* margin across all of a customer's 2019 purchasing, not the average of their transaction-level margins. Transaction-level margins aren't recoverable because the source data is aggregated to the quarter.

**Findings:** unlike the other four distributions, margin is **left-skewed**.

**Bins:** width 5%, plus a `< 0%` bin for loss-makers. The book keeps the (85,90], (90,95], (95,100] bins visible even though they're empty — your call.

### 1.7 Decile analyses
Two distinct constructions. **Present one or the other, never both** — pick based on the narrative you want.

#### (a) Customer decile report — each decile = 10% of *customers*
1. Sort customers by 2019 profit, descending.
2. Rank 1…N.
3. `decile = int(10 * (rank - 1) / N) + 1` — note the `rank - 1`, which is what keeps decile 1 from being off by one customer.
4. Aggregate by decile: customer count, total transactions, total spend, total profit.
5. Then apply the profit decomposition to produce the report columns:

| Column | Formula |
|---|---|
| % of customers | decile customers / total customers |
| % of transactions | decile trans / total trans |
| % of revenue | decile spend / total spend |
| % of profit | decile profit / total profit |
| Avg spend per customer | decile spend / decile customers |
| Avg profit per customer | decile profit / decile customers |
| AOF | decile trans / decile customers |
| AOV | decile spend / decile trans |
| Avg margin | decile profit / decile spend |

The bottom row (totals) gives the firm-level AOF/AOV/margin.

#### (b) Profit decile report — each decile = 10% of *profit*
1. Sort by profit descending, compute **cumulative profit**.
2. Decile boundaries in cumulative-profit space: `k × (total_profit / 10)` for k = 1…9.
3. For each boundary, find the individual customer profit value at the point the cumulative series first crosses it. These become the profit cut-offs.
4. Assign each customer to a decile by comparing their own profit against those cut-offs.
5. Aggregate and apply the same decomposition table as above.

**Watch out:** cumulative profit is **not monotonic**. It peaks at **$2,802,771.95** and then *declines* to the total of **$2,798,903.68**, because **263 customers are loss-making**. Decile 1 cut-off lands at ≈ **$545.53**, decile 2 at ≈ **$345.25**. Loss-makers all end up in decile 10.

*Variant worth building:* pull the loss-makers out as a separate 11th group rather than burying them in decile 10.

*Variant worth building:* if you don't have cost/profit data, run the same decile analysis on **revenue**.

---

## 2. Lens 2 — What changed between two periods? (2018 vs 2019)

**Objective:** decompose year-on-year change in firm performance into changes in customer behaviour.

### Working dataset
One row per customer active in **either** 2018 or 2019 (outer join of the two annual aggregates, zero-filled): `trans_2018, spend_2018, profit_2018, trans_2019, spend_2019, profit_2019`. **48,238 customers.**

Add a `years` status flag:
- `both` if active in both years
- `2018 only`
- `2019 only`

### 2.1 Headline
| | 2018 | 2019 | Δ |
|---|---|---|---|
| Revenue | $4,815,502 | $5,836,712 | +21% |
| Profit | $2,290,343 | $2,798,904 | +22% |
| Active customers | 26,254 | 31,855 | +21% |

### 2.2 Overlaid distributions
Re-run each Lens 1 distribution for 2018 and 2019 on the same axes, using **identical bin edges** and **relative frequencies** (the customer counts differ, so raw counts are not comparable).

**Finding:** the spend distribution for 2018 actives is very close to that for 2019 actives — the shape of customer heterogeneity is stable; the customer *count* is what moved. (Repeat for profit, transactions, avg spend per transaction, margin.)

### 2.3 Customer overlap
| Group | Customers |
|---|---|
| Active 2018 | 26,254 |
| Active 2019 | 31,855 |
| Active both years | 9,871 |
| 2018 only (lapsed) | 16,383 |
| 2019 only (new/reactivated) | 21,984 |

Repeat buyers are **38%** of 2018 actives and **31%** of 2019 actives.

**Area-proportional Venn diagram** (see Appendix A below for the geometry — it's a small numerical solve, easy in `scipy.optimize`).

### 2.4 Profit by activity group
Cross-tabulate `years` against 2018 profit and 2019 profit:

| Group | 2018 profit | 2019 profit |
|---|---|---|
| Both years | ≈ $1,216,349 | ≈ $1,183,785 |
| 2018 only | ≈ $1,073,946 | — |
| 2019 only | — | ≈ $1,615,125 |
| **Total** | **≈ $2,290,295** | **≈ $2,798,910** |

**Findings:** the both-years group produces **53% of 2018 profit but only 42% of 2019 profit**, and their profit *fell* by **≈ $32,618** year on year. Growth came entirely from acquisition. Repeat customers are over-represented in profit relative to their headcount share (38%/31%) — explain that gap with the AOF × AOV × Margin decomposition applied to each of the three groups (active customers, total trans, total spend, total profit → AOF, AOV, margin per group per year).

### 2.5 Decile change analysis
Do high-value customers stay high-value?

1. Compute **profit-decile boundaries separately for 2018 and 2019** (same cumulative-profit method as §1.7b).
2. Compare them — in this dataset they come out very close. So use a **common set of cut-offs for both years**: average the two years' thresholds and round to the nearest $5, giving:

   **545, 350, 255, 195, 150, 115, 90, 65, 40**

   Decile 1 = profit > $545; decile 2 = (350, 545]; … decile 10 = ≤ $40 (including all negatives).

3. Assign every customer a 2018 decile and a 2019 decile using those cut-offs. Customers inactive in a year get profit 0 → decile 10, which is harmless because you filter by `years` before tabulating.
4. Build a 2018-decile × 2019-decile cross-tab **restricted to the `both` group** (9,871 customers).
5. Append a **`2018 only` column** (counts by 2018 decile, 16,383 customers) and a **`2019 only` row** (counts by 2019 decile, 21,984 customers).
6. Row totals as % of 26,254; column totals as % of 31,855.

**Trade-off to be explicit about:** common cut-offs mean each decile no longer holds exactly 10% of each year's profit. Year-specific cut-offs preserve the 10%-of-profit property but make the transition matrix harder to read. Analyst's call.

**Finding to expect:** heavy mass on the diagonal and just below it, plus a very large `2018 only` column concentrated in the low deciles — i.e. churn is heaviest among low-value customers, but decile 1 is not immune.

### 2.6 Up-down analysis
For the 9,871 customers active in both years, decompose profit change into its drivers.

For each customer, build four binary flags (define **Up = 2019 value ≥ 2018 value**):
- `profit_up`
- `trans_up`
- `aspt_up` — average spend per transaction (`spend/trans`)
- `margin_up` — (`profit/spend`)

Concatenate into a 4-bit pattern → 16 possible groups. Aggregate: customer count, total 2018 profit, total 2019 profit, and profit change per group. Add rows for `2018 only` (all profit lost) and `2019 only` (all profit new), then a total.

**Findings and gotchas:**
- **One customer has zero 2019 spend → undefined margin.** Drop them (analysis then covers 9,870) or force them into the Down-Down-Down-Down group.
- Only **14 of 16 groups** appear in this 1% sample. **Up-Down-Down-Down** and **Down-Up-Up-Up** are missing — but they are *logically possible*, not impossible: a loss-making customer who becomes less unprofitable can have profit Up while all three drivers are Down, and vice versa. In the full dataset there are 29 and 38 such customers respectively. **Don't hard-code 14 groups.**
- **The `≥` in the Up definition matters a lot for transactions.** Of the 6,524 customers labelled Up on transactions, **3,273 had the *same* transaction count in both years** — a third of the repeat base. Profit ties are rare (4 customers); spend ties are rare (41). **Recommendation: use three levels (Up / Same / Down) for transactions**, two for the rest.
- Largest positive contribution comes from Up-Up-Up-Up (~1,836 customers, +$221k); largest negative from Down-Down-Down-Down (~931 customers, –$170k). But the dominant swing factors overall are the 2018-only (–$1.07M) and 2019-only (+$1.62M) blocks.

---

## 3. Lens 3 — How does a cohort evolve? (Q1/2016 cohort)

**Objective:** track one acquisition cohort across its lifetime. Cohort = customers whose first-ever purchase was in Q1/2016.

**Cohort size: 2,944 customers.** (Quarterly granularity means TCBA Figures 5.1 and 5.8, which need weekly/daily data, can't be reproduced.)

### 3.1 Working dataset A — quarterly cohort summary
Filter long data to `Cohort == 'y2016 q1'`, group by `YearQuarter`: number of active cohort members, total transactions, total spend, total profit. 16 rows.

### 3.2 Revenue decomposition over time
```
Revenue_t = cohort size × %active_t × ASPAC_t
ASPAC_t   = AOF_t × AOV_t
```
Plot each component by quarter:
- **Cohort revenue** — massive drop in the quarter *after* acquisition, then slow decline with a mild Q4 seasonal bump each year.
- **% active** — `active_t / cohort_size`. Same cliff.
- **ASPAC** — `spend_t / active_t`.
- **AOF** — `trans_t / active_t`. (In the 1% sample, AOF seasonality is much weaker than in the full dataset — a sampling artefact worth noting.)
- **AOV** — `spend_t / trans_t`.

The point of the decomposition: revenue decay is driven overwhelmingly by **% active** collapsing, not by spend-per-active-buyer eroding.

### 3.3 Annual repeat-buying patterns
Build a customer × quarter transaction matrix for the cohort, then collapse to four annual binary flags:

- **2016 flag** = 1 if the customer made **more than one** transaction in 2016 (i.e. at least one *repeat* purchase beyond the acquisition purchase). *This year is special — do not use `> 0`.*
- **2017 / 2018 / 2019 flags** = 1 if any transaction in that year.

Concatenate to a 4-bit pattern → 16 groups; report count and % of cohort per pattern, sorted descending.

**Headline finding: 45% of the Q1/2016 cohort never made a second purchase by the end of 2019.** The "always on" pattern (Y-Y-Y-Y) is ~8%.

Present the table with Y/N rather than 1/0 — audiences parse it faster.

### 3.4 Time to second purchase
From the same customer × quarter matrix, build a cumulative "has made a second-ever purchase" indicator:

- Quarter 1 (acquisition quarter): `1 if trans_q1 > 1 else 0`
- Quarter t > 1: `max(indicator_{t-1}, 1 if trans_t > 0 else 0)`

Column sums ÷ cohort size → **cumulative % of cohort having made a second purchase** by end of each quarter. First differences → **% making their second purchase *in* each quarter**.

### 3.5 Quarterly repeat-buying rate *(optional; not in TCBA)*
Definition: % of customers active in period *t* who are also active in period *t+1*.

Build a customer × quarter binary activity matrix (`trans > 0`), then:
```
RBR_t = Σ (active_t × active_{t+1}) / Σ active_t
```
(i.e. dot product of adjacent columns over the sum of the earlier column.)

**Conceptual distinction:** RBR is a period-to-period measure and is blind to within-period repeat purchases. A customer who bought five times in Q1/2016 and never again shows up as a repeat buyer in §3.4 but *not* in the RBR series. Both measures are needed.

### 3.6 Working dataset B — value to date (VTD)
Group the cohort's records by `CustomerID` across all 16 quarters: total transactions, total spend, total profit. **VTD = total (undiscounted) profit over 2016–2019.** 2,944 rows.

**Findings:** VTD ranges **–$23 to $3,756**. Mean **$170**, median **$78** → **72% of cohort members are below average**. 5th pct $5.56; 10th pct $12.05; top 5% above $663.09. Just over **2% have VTD > $1,000**. Total cohort VTD ≈ **$499,821**.

**Bins:** width $25, censor at $1,000.

### 3.7 VTD decile analysis
Same construction as the **profit decile report** (§1.7b), applied to VTD: each decile = 10% of the cohort's total VTD. Decile 1 cut-off ≈ **$1,373.43** (individual VTD).

Report per decile: % of cohort, % of transactions, % of spend, % of VTD, avg spend, avg VTD, AOF, AOV, margin.

**Key finding:** value concentration is driven by **frequency, not basket size**. AOF for decile 1 is ~**25×** decile 10's; AOV is only ~**2×**. High-value customers are high-value because they *come back*.

### 3.8 Annual % active by VTD decile
Cross the VTD decile label with the annual activity flags from §3.3 (but with 2016 set to 1 for everyone — by construction every cohort member was active in their acquisition year). Report, for each decile, the % of its members active in 2016, 2017, 2018, 2019.

This is the follow-through on §3.7: the top deciles stay alive; the bottom deciles vanish after year one.

### 3.9 RFM analysis
Computed at the end of the 4-year window, on the cohort.

- **R (recency)** = index of the last quarter with a transaction, 1 (Q1/2016) … 16 (Q4/2019).
- **F (frequency)** = total transactions over the four years.
- **M (monetary value)** = **average profit per transaction** = total profit / total transactions. *(Note: this is the book's definition. Other definitions exist — state yours.)*

Bins:
| Dim | Bins |
|---|---|
| R | 1 = Q1/2016 only; 2 = Q2/2016–Q4/2017 (q2–q8); 3 = Q1/2018–Q3/2019 (q9–q15); 4 = Q4/2019 |
| F | 1 = one purchase; 2 = 2–4; 3 = 5–10; 4 = 11+ |
| M | 1 = ≤ $25; 2 = ($25, $50]; 3 = ($50, $75]; 4 = > $75 |

Cross-tab: rows = R × M, columns = F.

**Structural note:** 4×4×4 = 64 cells, but only **52 are feasible** — a customer with F = 1 made their only purchase in the acquisition quarter, so **F = 1 implies R = 1**. Twelve cells (F=1 with R ∈ {2,3,4}) are structurally empty. Don't treat them as zeros to be explained.

Bin boundaries are judgment calls. The two that are near-mandatory: a **standalone F = 1 bin**, and **standalone recency bins for the first and last periods**.

---

## 4. Lens 4 — Comparing cohorts

**Objective:** compare and contrast the performance of different acquisition cohorts, controlling for cohort size.

### Working dataset
Four **cohort × quarter** matrices (17 quarterly cohorts incl. `pre 2016`, by 16 quarters):
1. number of active customers
2. total transactions
3. total spend
4. total profit

**Cohort size** = the diagonal of matrix (1) — the acquisition quarter's active count. **Exclude the `pre 2016` row** from anything requiring cohort size.

Then derive three more matrices:
- **% active** = active / cohort size *(quarterly cohorts only)*
- **AOF** = trans / active *(all cohorts, incl. pre-2016)*
- **AOV** = spend / trans
- **Avg margin** = profit / spend

### 4.1 Cohort comparison workflow (Q3/2016 vs Q4/2016)
Cohort sizes: **Q3/2016 = 2,842**, **Q4/2016 = 6,162** — more than 2× apart, so raw comparison is meaningless.

1. **Plot raw quarterly profit** for both cohorts. Q4 dominates simply because it's bigger.
2. **Index each cohort's profit to its own acquisition-quarter profit** (= 100). This normalises for size — but it still tells you *nothing about why* one decays faster than the other.
3. **Decompose.** Plot, for each cohort, on the same axes:
   - % cohort active by quarter
   - AOF by quarter
   - AOV by quarter
   - average margin by quarter

   This is where the actual insight lives. The decomposition tells you whether a cohort underperforms because fewer of them come back, because they come back less often, because they spend less per order, or because they buy lower-margin goods. Those are four completely different problems with four completely different responses.

4. Repeat for **Q4/2016 vs Q4/2017** — a like-for-like seasonal comparison across years.

---

## 5. Lens 5 — Health of the customer base

**Objective:** firm-level view. Are we growing because the base is healthy, or because we're outrunning churn with acquisition?

### Working datasets
Annual cohorts: **pre-2016, 2016, 2017, 2018, 2019**. Derive each customer's `CohortYear` from their `Cohort` label (keep `pre y2016` as its own level; otherwise take the year part).

Four **annual cohort × calendar year** matrices (5 × 4):
1. **Number active** — count of cohort members with ≥1 transaction in that year. *(This is the awkward one: build a customer × year activity indicator first, then group by cohort year and sum.)*
2. **Total transactions**
3. **Total spend**
4. **Total profit**

Also keep a per-customer table: `CustomerID, trans_2016..trans_2019, CohortYear` — several later analyses need it.

**Validation targets — active customers by year:** 2016 = 20,673; 2017 = 21,434; 2018 = 26,254; 2019 = 31,855.

**Validation targets — total profit by year:** 2016 = $1,871,911; 2017 = $1,953,229; 2018 ≈ $2,290,343; 2019 ≈ $2,798,904.

### 5.1 Annual performance
- Bar chart: annual revenue and profit.
- **Stacked bar: annual profit by acquisition cohort.** This is the single most important picture in the audit.
- Bar chart: **new customers acquired each year** (the diagonal of matrix 1).

### 5.2 The two percentages that matter
For the profit-by-cohort stack, annotate two different ratios — they answer different questions.

**(a) Share of a year's profit from that year's new customers.**
2016: $1,193,524 / $1,871,911 = **64%** → 36% of 2016 profit came from customers acquired earlier.

**(b) Year-on-year retention of profit from existing cohorts.**
Profit in 2017 from all cohorts acquired *before* 2017 = $1,953,229 – $964,671 = $988,558.
That's **53%** of what those same cohorts delivered in 2016 ($1,871,911).

And per-cohort:
- The 2016 cohort delivered $1,193,524 in 2016 and $451,670 in 2017 → **38%** retention of profit.
- The pre-2016 cohort delivered $678,387 in 2016 and $536,888 in 2017 → **79%** retention.

**The story:** new cohorts decay fast (38% in year two); old, self-selected surviving cohorts are far stickier (79%). Growth in total profit is being bought with acquisition, while the profit contributed by each existing cohort falls roughly by half annually. Repeat the same annotation for the active-customer stack.

### 5.3 Time to second purchase, by annual cohort
Cumulative % of each annual cohort that has made a second-ever purchase by end of each year.

Logic per customer × year cell:
- 0 if the year precedes the cohort year
- if year == cohort year: `1 if trans > 1 else 0` (repeat beyond the acquisition purchase)
- if year > cohort year: `max(previous_flag, 1 if trans > 0 else 0)`

Sum by cohort year, divide by cohort size. **Exclude the `pre 2016` cohort** — its size is unknown and its "acquisition year" is outside the window.

Result is a triangular table (each cohort's series starts in its acquisition year).

### 5.4 Annual repeat-buying rate
```
RBR(cohort, x→x+1) = # cohort members active in BOTH year x and x+1
                     ÷ # cohort members active in year x
```
Build pairwise activity indicators (16/17, 17/18, 18/19) at the customer level, sum by cohort year, then divide by the corresponding column of matrix (1).

Report per cohort **and** overall (all customers, not split by cohort).

Rows: pre-2016, 2016, 2017, 2018, Overall. **There is no 2019 row** — you'd need 2020 data. Expect the pre-2016 cohort's RBR to be substantially higher than any new cohort's: survivorship, not superiority.

### 5.5 Full cohort decomposition by year
For each annual cohort × year:

| Table | Formula |
|---|---|
| **% active** | active / cohort size *(2016–2019 cohorts only; NaN for pre-2016)* |
| **Avg annual profit per active member** | profit / active |
| **Annual AOF** | trans / active |
| **Annual AOV** | spend / trans |
| **Annual avg margin** | profit / spend |

Plot each as a line chart, one line per cohort. Add a **Total** row (all customers) to each — the totals row of the last three gives you the **overall AOF, AOV and margin by year**, which is the firm-level summary of whether the business is changing shape.

**Note on plotting:** cells before a cohort exists must be **missing (NaN), not zero** — a zero will be plotted and will distort the line.

### 5.6 Quarterly version
Same three pictures, at quarterly granularity using quarterly cohorts (reuse the Lens 4 matrices):
1. Quarterly revenue and profit (column totals of the spend and profit matrices).
2. Quarterly profit **stacked by quarterly cohort** — the fine-grained version of §5.1.
3. Number of customers acquired each quarter (the diagonal).

---

## Appendix A — Area-proportional Venn diagram (Lens 2)

Excel can't draw this; matplotlib can (or use `matplotlib_venn`, though solving it yourself is trivial and gives you exact control).

Given: 2018 actives = 26,254; 2019 actives = 31,855; both = 9,871.

Set the 2018 circle radius **R = 1**. Then, requiring areas proportional to counts:

```
r = sqrt(31855 / 26254) = 1.1015
```

Required overlap area (in the same normalised units):

```
A = π × (9871 / 26254) = 1.1812
```

Solve numerically for the centre-to-centre distance **d** satisfying the standard circle-circle intersection area:

```
A(d) = r² · arccos( (d² + r² − R²) / (2dr) )
     + R² · arccos( (d² − r² + R²) / (2dR) )
     − ½ · sqrt( (d + r − R)(d − r + R)(−d + r + R)(d + r + R) )
```

Minimise `(A_target − A(d))²` over d (any 1-D root-finder / `scipy.optimize.brentq` on `A(d) − A_target`). **Solution: d = 1.1469.**

Draw: circle radius 1 centred at `(1, max(1, r))`; circle radius r centred at `(1 + d, max(1, r))`.

---

## Appendix B — Implementation checklist

- [ ] Load long CSV once; derive `Year`; keep as the single source of truth.
- [ ] Write one reusable `describe_and_bin(series, width, cutoff, neg_bin=False)` helper — six of the histograms in Lens 1/3 are the same function with different arguments.
- [ ] Write one reusable `decile_report(df, value_col, method='customer'|'value')` — Lens 1 uses it twice, Lens 3 once.
- [ ] Write one reusable `decompose(df_grouped)` returning %cust / %trans / %spend / %profit / avg spend / avg profit / AOF / AOV / margin. Used in Lens 1, 2, 3, 4, 5.
- [ ] Guard every ratio against zero denominators (`Spend == 0` → margin undefined; `NumTrans == 0` → ASPT undefined). Two customers in 2019, one in the Lens 2 both-group. Do not silently zero-fill.
- [ ] Keep pre-2016 cohort in transaction/spend/profit aggregates but **out of** every cohort-size-denominated ratio.
- [ ] Use NaN (not 0) for cohort-year cells that pre-date the cohort's existence.
- [ ] Cross-check totals at every stage: 31,855 / 60,730 / $5.84M / $2.80M for 2019; 48,238 customers in the Lens 2 frame; 2,944 in the Q1/2016 cohort.